# Определение ориентации текстового кропа (0° / 180°)

## Задача
Бинарная классификация: определить, повёрнут ли кроп с текстом на 180°.

## Подход

**Модель**: `PP-LCNet_x1_0_textline_ori` (Apache 2.0, PaddleOCR).
Дообучение головы на собственной синтетике (~70к пар).

**Ключевые решения генератора **:
- Независимая генерация примеров.
- Тёмные фоны (30%) + светлые.
- Реалистичные боксы детектора: асимметричные поля, обрезка краёв.
- Наклон ±5–8° (вместо ±15–30°, которые ломали обучение).
- JPEG-артефакты, шум, градиенты, размытие.

**Валидация**: псевдо-валидация на 2000 реальных тестовых кропах.
Для пары `(X, rot(X))` считаем symmetric 1 − Brier (min по двум гипотезам),
`rot_consistency` и долю уверенно различаемых пар.

## Результаты

| Модель | Псевдо | Тест |
|---|---|---|
| x1(v4) | **0.9771** | **0.9562** |
| ансамбль x0_25+x1 | 0.9760 | — |
| x0_25(v4) | 0.9464 | — |

**Финальное решение**: x1(v4), `outputs/best_head_v4_x1.pth`.

## Воспроизводимость
- Веса `weights/x1/` — PaddleOCR (Apache 2.0). (веса взяты с https://huggingface.co/PaddlePaddle/PP-LCNet_x1_0_textline_ori_safetensors/tree/main )
- Seed = 2026.
- Сетевые вызовы в рантайме отсутствуют.

In [10]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForImageClassification


SEED = 2026

def set_seed(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [11]:
# ===== загрузка x1-модели =====
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# >>> каталог с весами x1
WEIGHTS_DIR = "weights/x1"

model = AutoModelForImageClassification.from_pretrained(
    WEIGHTS_DIR, local_files_only=True
)
# Накладываем дообученные веса .
state = torch.load("outputs/best_head_v3_x1.pth", map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")

model.to(DEVICE).eval()
processor = AutoImageProcessor.from_pretrained(WEIGHTS_DIR, local_files_only=True)

print(f"Loaded: {model.config._name_or_path}")
print(f"Num labels: {model.config.num_labels}")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Missing keys: 0, Unexpected: 0
Loaded: weights/x1
Num labels: 2


In [12]:
# ===== инференс =====
USE_TTA = False

@torch.no_grad()
def predict_batch(images, model, processor, device, use_tta=USE_TTA):
    """Единый путь инференса. images: list[PIL.Image] в RGB."""
    inp = processor(images, return_tensors="pt").to(device)
    logits = model(**inp).last_hidden_state
    probs = F.softmax(logits, dim=-1)[:, 1]

    if use_tta:
        imgs_rot = [im.rotate(180) for im in images]
        inp_r = processor(imgs_rot, return_tensors="pt").to(device)
        logits_r = model(**inp_r).last_hidden_state
        probs_r = F.softmax(logits_r, dim=-1)[:, 1]
        probs = 0.5 * (probs + (1.0 - probs_r))

    return probs.cpu().numpy()

In [13]:
# ===== псевдо-валидация =====
# Метки на тесте неизвестны. Используем пары (X, rot(X)):
# для любой ориентации X одна из двух гипотез верна, поэтому
# symmetric Brier = min(brier1, brier2) — корректная оценка без меток.
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

VAL_N = 2000
ids = pd.read_csv("sample_submission.csv")["image_id"].head(VAL_N).tolist()

imgs_orig, imgs_rot = [], []
for tid in tqdm(ids, desc="Building val"):
    for ext in [".png", ".jpg", ".jpeg"]:
        p = Path("test_images") / f"{tid}{ext}"
        if p.exists():
            img = Image.open(p).convert("RGB")
            imgs_orig.append(img)
            imgs_rot.append(img.rotate(180))
            break

p_orig, p_rot = [], []
for i in range(0, len(imgs_orig), 64):
    p_orig.extend(predict_batch(imgs_orig[i:i+64], model, processor, DEVICE))
    p_rot.extend( predict_batch(imgs_rot[i:i+64],  model, processor, DEVICE))
p_orig = np.array(p_orig)
p_rot  = np.array(p_rot)

rot_consistency = np.abs(p_orig + p_rot - 1).mean()
brier1 = (p_orig - 0)**2 + (p_rot - 1)**2
brier2 = (p_orig - 1)**2 + (p_rot - 0)**2
brier_sym = np.minimum(brier1, brier2).mean() / 2
confident = (np.abs(p_orig - p_rot) > 0.8).mean()

print(f"rot_consistency:          {rot_consistency:.4f}")
print(f"Symmetric 1 - Brier:      {1 - brier_sym:.4f}")
print(f"Уверенно различает пар:   {confident:.2%}")

Building val: 100%|██████████| 2000/2000 [00:01<00:00, 1182.29it/s]


rot_consistency:          0.0639
Symmetric 1 - Brier:      0.9771
Уверенно различает пар:   89.40%


In [14]:
# ===== Батчевый инференс на 20 000 =====
TEST_DIR = Path("test_images")
BATCH_SIZE = 64

sample = pd.read_csv("sample_submission.csv")
image_ids = sample["image_id"].tolist()
probs = np.full(len(image_ids), 0.5, dtype=np.float32)
missing = []

def find_image(image_id):
    for ext in [".png", ".jpg", ".jpeg"]:
        p = TEST_DIR / f"{image_id}{ext}"
        if p.exists():
            return p
    return None

buf_imgs, buf_idx = [], []
def flush():
    if not buf_imgs:
        return
    batch_probs = predict_batch(buf_imgs, model, processor, DEVICE)
    for idx, p in zip(buf_idx, batch_probs):
        probs[idx] = p
    buf_imgs.clear()
    buf_idx.clear()

for i, image_id in enumerate(tqdm(image_ids, desc="Inference")):
    path = find_image(image_id)
    if path is None:
        missing.append(image_id)
        continue
    buf_imgs.append(Image.open(path).convert("RGB"))
    buf_idx.append(i)
    if len(buf_imgs) == BATCH_SIZE:
        flush()
flush()

if missing:
    print(f"Не найдено: {len(missing)}")
    print(missing[:10])

Inference: 100%|██████████| 20000/20000 [00:37<00:00, 531.76it/s]


In [16]:
submission = pd.DataFrame({
    "image_id": sample["image_id"],
    "p_180": probs,
})

# Проверки формата
assert len(submission) == 20000
assert submission["p_180"].between(0, 1).all()
assert submission["p_180"].notna().all()

submission.to_csv("submission.csv", index=False)
print(f"Сохранено: submission.csv ({len(submission)} строк)")
print(submission.head())

Сохранено: submission.csv (20000 строк)
     image_id         p_180
0  test_00000  1.941633e-07
1  test_00001  1.000000e+00
2  test_00002  1.000000e+00
3  test_00003  1.853168e-02
4  test_00004  9.999664e-01


## Итог

- Submission сохранён в `submission.csv` (20000 строк)
- Модель: PP-LCNet_x0_25_textline_ori (0.96 MB, Apache 2.0)
- Инференс: batch=64, без TTA
- Веса загружены локально, сетевых вызовов нет
- Seed = 2026, воспроизводимость обеспечен

Оригинальная голова PP-LCNet_x1_0_textline_ori от PaddleOCR обучалась преимущественно на китайском и английском — кириллица в ней представлена слабо. На псевдо-валидации модель сомневалась именно на кропах с русским текстом: буквы В, Н, Р, С, У, Х при повороте дают другие формы или похожие буквы, а шрифтов русских вывесок в обучающих данных почти нет. Дообучение головы на синтетике с кириллицей и латиницей закрыло этот разрыв.

Генератор дорабатывался по артефактам кропов, на которых модель сомневалась (|p(X) − p(rot(X))| < 0.8): сохраняли такие пары, смотрели глазами, выявляли общий паттерн, добавляли в генератор.

### Метрики валидации (2000 тестовых кропов, псевдо-val)

- Symmetric 1 − Brier: 0.9366
- rot_consistency: 0.1651
- Доля уверенно различаемых пар: 68.80%